<a href="https://colab.research.google.com/github/marcusvmingoransi/mdm-seats-aero-fullstack/blob/main/AMM2_Atividade_Modular_do_M%C3%B3dulo_2_Cap%C3%ADtulo_8_Redes_Neurais_Artificiais.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score

In [11]:
df = pd.read_csv('data.csv')

In [12]:
df

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,926424,M,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,NaN
565,926682,M,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,NaN
566,926954,M,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,NaN
567,927241,M,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,NaN


In [13]:
X = df.iloc[:, 2:-1]
y = df.iloc[:, 1]
y = y.replace({'B': 0, 'M': 1})

<ipython-input-13-0d44681c007f>:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = y.replace({'B': 0, 'M': 1})


In [14]:
# Para a divisão em treino e teste, vamos separar 70% para treino e 30% para teste, declarando o seed random.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=10)

In [15]:
# Pergunta 9
# Em nosso problema prático, quais as dimensões dos datasets X_train, X_test, y_train, y_test após a divisão em treino e teste (formato linhas x colunas):

print("Dimensões de X_train:", X_train.shape)
print("Dimensões de X_test:", X_test.shape)
print("Dimensões de y_train:", y_train.shape)
print("Dimensões de y_test:", y_test.shape)

Dimensões de X_train: (398, 30)
Dimensões de X_test: (171, 30)
Dimensões de y_train: (398,)
Dimensões de y_test: (171,)


In [16]:
# Para a aplicação do PCA é importante lembrar que primeiro devemos aplicar o Standard Scaler, usando fit_transform na base de treino, mas apenas o
# transform na base de teste

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [17]:
# A seguir é possível aplicar a transformação com PCA nos dados já escalonados:

pca = PCA(n_components=10, random_state=10)
pca.fit(X_train_scaled)

PCA(n_components=10, random_state=10)

In [18]:
# Após isso, será possível analisar a explicabilidade das variâncias nos dados, a partir dos componentes:

pca.explained_variance_ratio_

array([0.4383292 , 0.18413908, 0.10158532, 0.06803093, 0.05624249,
       0.03953245, 0.02437355, 0.01630403, 0.01462998, 0.01175767])

In [19]:
# Pergunta 10
# prompt: Em nosso problema prático, após aplicar o PCA, pelo menos quantos componentes representam 70% da variância dos dados:

import numpy as np

variancia_acumulada = np.cumsum(pca.explained_variance_ratio_)
n_componentes = np.argmax(variancia_acumulada >= 0.7) + 1
print("Número mínimo de componentes que explicam pelo menos 70% da variância:", n_componentes)


Número mínimo de componentes que explicam pelo menos 70% da variância: 3


In [20]:
# Em nosso problema, queremos criar os modelos de classificação utilizando apenas os primeiros componentes que representem pelo menos 70% da
# variância dos dados. Para isso você deve analisar as variâncias das componentes. Feita a análise, criamos então os datasets X_train_pca e
# X_test_pca, transformados pelo PCA e utilizando o número de componente identificado por você, em n_components.

pca = PCA(n_components=0.7, random_state=10)
pca.fit(X_train_scaled)
X_train_pca = pca.transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [21]:
# Pergunta 11
# Em nosso problema prático, após identificar quantos componentes representam 70% da variância dos dados, e aplicar a transformação PCA nos dados padronizados, quais as dimensões do X_train_pca, X_test_pca (formato linhas x colunas):

print("Dimensões de X_train_pca:", X_train_pca.shape)
print("Dimensões de X_test_pca:", X_test_pca.shape)


Dimensões de X_train_pca: (398, 3)
Dimensões de X_test_pca: (171, 3)


In [22]:
# Você deve definir qual o tipo de Busca irá aplicar: GridSearch ou RandomSearch. A busca deve ser realizada utilizando-se validação cruzada
# com 5 partições e a acurácia (‘accuracy’) como métrica. O seed random do modelo deve ser declarado:

# Definindo os hiperparâmetros a serem testados (exemplo)
param_distributions = {
    'max_depth': [3, None],
    'min_samples_split': [2, 10]
}

estimator = DecisionTreeClassifier(random_state=10)
# Criando o objeto de busca (RandomizedSearchCV)
grid = RandomizedSearchCV(
    estimator=estimator,
    param_distributions=param_distributions,  # Parâmetro para RandomSearch
    scoring='accuracy',
    cv=5,
    random_state=10
)

grid.fit(X_train_pca, y_train)

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 4 is smaller than n_iter=10. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


RandomizedSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=10),
                   param_distributions={'max_depth': [3, None],
                                        'min_samples_split': [2, 10]},
                   random_state=10, scoring='accuracy')

In [29]:
# grid.cv_results_
# grid.best_index_
grid.best_params_
# grid.best_score_
# grid.best_estimator_

{'min_samples_split': 2, 'max_depth': 3}

In [30]:
# prompt: Vamos aplicar o método Bagging, utilizando como modelo base nossa árvore
# de decisão com os melhores hiperparâmetros encontrados na etapa anterior.
# Portanto será necessário criar um modelo base, com os melhores
# hiperparâmetros:

# Criando o modelo base com os melhores hiperparâmetros
best_tree_model = grid.best_estimator_

# Aplicando o Bagging
bagging_model = BaggingClassifier(estimator=best_tree_model, n_estimators=10, random_state=10)
bagging_model.fit(X_train_pca, y_train)

# Fazendo previsões com o modelo Bagging
y_pred_bagging = bagging_model.predict(X_test_pca)

# Avaliando a acurácia do modelo Bagging
accuracy_bagging = accuracy_score(y_test, y_pred_bagging)
print(f"Acurácia do modelo Bagging: {accuracy_bagging}")



Acurácia do modelo Bagging: 0.9532163742690059


In [32]:
# Vamos aplicar o método Bagging, utilizando como modelo base nossa árvore
# de decisão com os melhores hiperparâmetros encontrados na etapa anterior.
# Portanto será necessário criar um modelo base, com os melhores
# hiperparâmetros:

best_max_depth = grid.best_params_['max_depth']
best_min_samples_split = grid.best_params_['min_samples_split']

# Criando o modelo com os melhores hiperparâmetros
estimator = DecisionTreeClassifier(random_state=10, max_depth=best_max_depth, min_samples_split=best_min_samples_split)


estimator DecisionTreeClassifier(max_depth=3, random_state=10)


In [33]:
# Construir o modelo ensemble com 100 estimadores:

ensemble = BaggingClassifier(estimator=estimator, n_estimators=100, random_state=10)
ensemble.fit(X_train_pca, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=3,
                                                   random_state=10),
                  n_estimators=100, random_state=10)

In [36]:
# Pergunta 13
# prompt: Como achar a acurácia média da melhor combinação de hiperparâmetros (mean_test_score)

print(f"Acurácia média da melhor combinação de hiperparâmetros (mean_test_score): {grid.best_score_}")


Acurácia média da melhor combinação de hiperparâmetros (mean_test_score): 0.9219620253164557


In [34]:
# Pergunta 14
grid.best_params_

{'min_samples_split': 2, 'max_depth': 3}

In [37]:
# Pergunta 15
# prompt: após aplicar o método ensemble Bagging, usando a árvore de decisão com os melhores hiperparâmetros, temos uma acurácia entre o y_pred e y_test de (arredondando para 6 casas decimais):

# Fazendo previsões com o modelo Ensemble
y_pred = ensemble.predict(X_test_pca)

# Avaliando a acurácia do modelo Ensemble
accuracy_ensemble = accuracy_score(y_test, y_pred)
print(f"Acurácia do modelo Ensemble: {accuracy_ensemble:.6f}")


Acurácia do modelo Ensemble: 0.959064
